In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    
# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDaC/notebooks


## Env

In [2]:
from src.utils.env import prepare_environment

prepare_environment("../api_keys")

INFO 08-19 22:47:32 [src.utils.env:30] Setting API tokens: ['WANDB_API_KEY', 'OPENPIPE_API_KEY', 'OPENAI_API_KEY']
INFO 08-19 22:47:32 [src.utils.env:42] Setting additional variables: {'NCCL_CUMEM_ENABLE': '0', 'VLLM_USE_V1': '0', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn', 'ART_SERVER_TIMEOUT': '300'}


## Model

In [4]:
import src.configs.models.art as art_configs
import src.configs.models.vllm as vllm_configs
from src.configs import PathConfig

print("Available ART model configurations:")
print(art_configs.available_configs())

print("Available vLLM model configurations:")
print(vllm_configs.available_configs())

base_model = "unsloth/Qwen3-14B"
project_name = "easy2hard_dac"

path_config = PathConfig(
    base_model=base_model,
    project_name=project_name,
)

Available ART model configurations:
['unsloth/Llama-4-Scout-17B-16E-Instruct', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Qwen2-7B', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen3-14B']
Available vLLM model configurations:
['unsloth/Llama-4-Scout-17B-16E-Instruct', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Qwen2-7B', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen3-14B', 'unsloth/Qwen3-32B', 'unsloth/Qwen3-32B-unsloth-bnb-4bit']


In [5]:
import art

host = "0.0.0.0"
port = 8200

model = art.Model(
    name=base_model,
    project=path_config.project_name,
    inference_api_key=os.getenv("OPENAI_API_KEY", "default"),
    inference_base_url=f"https://{host}:{port}/v1",
)

## Data

In [6]:
from datasets import Dataset, load_dataset, DatasetDict

dataset_dict: DatasetDict = load_dataset(
    path="furonghuang-lab/Easy2Hard-Bench",
    name="E2H-AMC",
    split=None,
)  # type: ignore

train_data: Dataset = dataset_dict["train"]
test_data: Dataset = dataset_dict["eval"]

## Inference Clients

In [7]:
from src.vllm_client import VllmClient, VllmRouter

inference_clients: list[VllmClient] = []
vllm_server_ports = [8200]
for port in vllm_server_ports:
    inference_clients.append(VllmClient.from_connection(port=port, base_model=base_model))

vllm_router: VllmRouter = VllmRouter(inference_clients)

## Config 

In [8]:
from experiments.easy2hard_tools.trainer import Easy2HardToolTrainer
from src.configs import PromptConfig, StopCriteria

prompt_config = PromptConfig(
    system_root="dac_sys_prompt_gilad_v2_root_tools",
    system_inter="dac_sys_prompt_gilad_v2_inter_tools",
    system_leaf="dac_sys_prompt_gilad_v2_leaf_tools",
)

stop_criteria = StopCriteria(max_depth=1)

trainer = Easy2HardToolTrainer(
    model=model,
    vllm_router=vllm_router,
    path_config=path_config,
    prompt_config=prompt_config,
    stop_criteria=stop_criteria,
)

### Inference Test

In [9]:
import random

for i in range(5):
    
    idx = random.randint(0, len(train_data) - 1)
    # idx = 228
    print(f"Selected random index: {idx}")
    sample = train_data[idx]
    problem = sample["problem"]
    true_answer = sample["answer"]

    print(f"Problem: {problem}")
    print(f"Answer: {true_answer}")
    print("-" * 200)
    print()

    preds = await trainer.predict([sample], verbose=True, seed=0, extra_body={"chat_template_kwargs": {"enable_thinking": False}}, stream=False)

# preds = await trainer.predict(train_data.to_list()[:10], verbose=True, extra_body={"chat_template_kwargs": {"enable_thinking": False}})

Selected random index: 326
Problem: In the rectangular parallelepiped shown, $AB$ = $3$, $BC$ = $1$, and $CG$ = $2$. Point $M$ is the midpoint of $\overline{FG}$. What is the volume of the rectangular pyramid with base $BCHE$ and apex $M$? [asy] size(250); defaultpen(fontsize(10pt)); pair A =origin; pair B = (4.75,0); pair E1=(0,3); pair F = (4.75,3); pair G = (5.95,4.2); pair C = (5.95,1.2); pair D = (1.2,1.2); pair H= (1.2,4.2); pair M = ((4.75+5.95)/2,3.6); draw(E1--M--H--E1--A--B--E1--F--B--M--C--G--H); draw(B--C); draw(F--G); draw(A--D--H--C--D,dashed); label("$A$",A,SW); label("$B$",B,SE); label("$C$",C,E); label("$D$",D,W); label("$E$",E1,W); label("$F$",F,SW); label("$G$",G,NE); label("$H$",H,NW); label("$M$",M,N); dot(A); dot(B); dot(E1); dot(F); dot(G); dot(C); dot(D); dot(H); dot(M); label("3",A/2+B/2,S); label("2",C/2+G/2,E); label("1",C/2+B/2,SE); [/asy]
Answer: 2
--------------------------------------------------------------------------------------------------------------

predict:   0%|          | 0/1 [00:00<?, ?it/s]

Role:
SYSTEM
Content:
You are a highly capable and truthful AI assistant that excels at logical reasoning.

When encountering complex tasks, you may break them down into smaller, manageable sub-tasks. When you do so, these sub-tasks will be assigned to sub-agent to solve and the answer is then immediately reported back to you. 

There are strict formatting rules which you must follow:

Your Turn Options:
- You may reason, and then create a sub-task by calling the tool `call_sub_agent`.
- You may reason, and then provide a final answer within <answer> </answer> block, which ends the conversation.

Sub-Task Requirements:
- Each sub-task must be fully self-contained, include all context, instructions, and expected output detail level.
- Only the arguments passed to the `call_sub_agent` function are received by the sub-agent as input.
- The sub-agent does not retain any conversational history at all, so every sub-task must include the full context and information necessary, including any r

Role:
ASSISTANT
Content:
To determine the volume of the rectangular pyramid with base $BCHE$ and apex $M$, we need to calculate the area of the base $BCHE$ and then use the formula for the volume of a pyramid: 
$$
\text{Volume} = \frac{1}{3} \times \text{Area of base} \times \text{Height}
$$

### Step 1: Determine the coordinates of points
We assume the rectangular parallelepiped has the vertices with the following coordinates based on the given values:
- Let $A = (0, 0, 0)$
- Then $B = (3, 0, 0)$
- $C = (3, 1, 0)$
- $D = (0, 1, 0)$
- $E = (0, 0, 2)$
- $F = (3, 0, 2)$
- $G = (3, 1, 2)$
- $H = (0, 1, 2)$

Point $M$ is the midpoint of $\overline{FG}$. So we calculate the midpoint of $FG$:
$$
M = \left( \frac{3 + 3}{2}, \frac{0 + 1}{2}, \frac{2 + 2}{2} \right) = (3, 0.5, 2)
$$

### Step 2: Find the area of the base
The base $BCHE$ is a rectangle. We calculate the length and width of the base:
- $BC = 1$
- $BE = 3$, and since $BE$ is vertical (as $E$ lies directly above $A$), then $BE$ is 

predict:   0%|          | 0/1 [00:00<?, ?it/s]

Role:
SYSTEM
Content:
You are a highly capable and truthful AI assistant that excels at logical reasoning.

When encountering complex tasks, you may break them down into smaller, manageable sub-tasks. When you do so, these sub-tasks will be assigned to sub-agent to solve and the answer is then immediately reported back to you. 

There are strict formatting rules which you must follow:

Your Turn Options:
- You may reason, and then create a sub-task by calling the tool `call_sub_agent`.
- You may reason, and then provide a final answer within <answer> </answer> block, which ends the conversation.

Sub-Task Requirements:
- Each sub-task must be fully self-contained, include all context, instructions, and expected output detail level.
- Only the arguments passed to the `call_sub_agent` function are received by the sub-agent as input.
- The sub-agent does not retain any conversational history at all, so every sub-task must include the full context and information necessary, including any r

CancelledError: 

    Role:
    ASSISTANT
    Content:
    To solve the given problem, we use geometric reasoning, angle relationships, and triangle similarity.
    
    ---
    
    We are given:
    
    - Triangle $ ABC $, where $ AB = 10 $, $ AC = 12 $
    - Point $ D $ is the intersection of the external angle bisector of $ \angle BAC $ with segment $ BC $
    - Point $ E $ is on ray $ AC $ such that $ \angle BDE = 2 \angle ADB $
    - $ CE = 33 $, and we are to find the ratio $ \frac{DB}{DE} $
    
    ---
    
    ### Step 1: Understand the Geometry and Setup
    
    We are told $ D $ lies on line $ BC $ such that it lies on the external angle bisector of $ \angle BAC $. Therefore, by the **external angle bisector theorem**, the ratio of the segments of line $ BC $ is:
    
    $$
    \frac{BD}{DC} = \frac{AB}{AC} = \frac{10}{12} = \frac{5}{6}
    $$
    
    ---
    
    ### Step 2: Consider the Location of Point $ E $
    
    We are told $ E $ is on **ray $ AC $** such that $ \angle BDE = 2 \

In [10]:
from src.configs import TrainingConfig
from src.utils.io import load_base_model
 
train_config = load_base_model(TrainingConfig, path="../experiments/easy2hard_tools/defaults/train_config.json", do_raise=True)

await trainer.train(
    train_config,
    train_dataset=train_data.to_list(),
    val_dataset=test_data.to_list(),
)

INFO 08-19 22:53:06 [src.utils.io:76] Loaded TrainingConfig from '../experiments/easy2hard_tools/defaults/train_config.json'.


ValueError: Model must be an `art.TrainableModel` to train.